In [ ]:
import os

REPO_URL = "https://github.com/meriem200512365/Chat-boot-cegedim.git"
REPO_DIR = "Chat-boot-cegedim"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}


In [ ]:
%pip install -q -r requirements.txt

## 3.2 (Re)indexation

Génère `data/chroma_db/` à partir de `data/menu.xml` — nécessaire une fois
par session Colab (l'environnement est jetable, `data/chroma_db/` n'est pas
versionné dans le dépôt).

In [ ]:
!python scripts/index_data.py

## 3.3 Import du vrai module de recherche du projet

In [ ]:
import sys
sys.path.insert(0, ".")

from src.search.semantic_search import search
from src.chatbot.chatbot import repondre


## 3.4 Jeu de questions test avec vérité terrain

Même logique que dans le notebook 2 : liste `(question, id_attendu)`.
Complète-la avec le maximum de cas réels (idéalement des questions
posées par de vrais utilisateurs, ou au minimum plusieurs formulations
différentes des fonctionnalités les plus utilisées).

In [ ]:
QUESTIONS_TEST = [
    ("je veux gerer les actes RO", "ID_A_COMPLETER_1"),
    ("comment parametrer un devis web", "ID_A_COMPLETER_2"),
    ("ou trouver la gestion des cheques", "ID_A_COMPLETER_3"),
    ("annuler un cheque", "ID_A_COMPLETER_4"),
    ("acces aux referentiels de prestations", "ID_A_COMPLETER_5"),
    # ... complete avec les ids reels trouves dans menu_index.json
    # (voir la recherche par mot-cle du notebook 2, section 2.3)
]


## 3.5 Calcul de precision@1, precision@3 et MRR

In [ ]:
def evaluer(questions_test, top_k=3):
    lignes = []
    for question, id_attendu in questions_test:
        resultats = search(question, top_k=top_k)
        ids_trouves = [r["id"] for r in resultats]

        precision_1 = int(len(ids_trouves) > 0 and ids_trouves[0] == id_attendu)
        precision_k = int(id_attendu in ids_trouves)

        rang = None
        if id_attendu in ids_trouves:
            rang = ids_trouves.index(id_attendu) + 1
        reciprocal_rank = 1 / rang if rang else 0

        lignes.append({
            "question": question,
            "id_attendu": id_attendu,
            "top1_trouve": ids_trouves[0] if ids_trouves else None,
            "precision@1": precision_1,
            f"precision@{top_k}": precision_k,
            "reciprocal_rank": reciprocal_rank,
        })
    return lignes

resultats_eval = evaluer(QUESTIONS_TEST, top_k=3)


In [ ]:
import pandas as pd

df_eval = pd.DataFrame(resultats_eval)
df_eval


In [ ]:
n = len(df_eval)
print(f"Nombre de questions evaluees : {n}")
print(f"Precision@1 : {df_eval['precision@1'].mean():.1%}")
print(f"Precision@3 : {df_eval['precision@3'].mean():.1%}")
print(f"Mean Reciprocal Rank (MRR) : {df_eval['reciprocal_rank'].mean():.3f}")


## 3.6 Analyse des erreurs

Pour chaque question mal répondue, on regarde ce que le système a trouvé à
la place — utile pour identifier des patterns d'erreur (synonymes non
capturés, ambiguïté métier, etc.).

In [ ]:
erreurs = df_eval[df_eval["precision@1"] == 0]
print(f"{len(erreurs)} question(s) sur {n} mal repondue(s) en top-1 :\n")

for _, row in erreurs.iterrows():
    print(f"Q: {row['question']}")
    print(f"   attendu : {row['id_attendu']}")
    print(f"   trouve  : {row['top1_trouve']}")
    print()


## 3.7 Test de bout en bout via `chatbot.repondre()`

Au-delà du simple ranking, on vérifie aussi le comportement du chatbot
complet (gestion de la confiance / ambiguïté / aucun résultat) sur les
mêmes questions.

In [ ]:
for question, _ in QUESTIONS_TEST:
    rep = repondre(question)
    print(f"Q: {question}")
    print(f"   type: {rep['type']}  |  {rep['message']}")
    print()
